# 02 - Eksperimen & Komparasi 5 Model (*From Scratch* vs `scikit-learn`)
**Penanggung Jawab:** Kolektif Seluruh Anggota (Anggota 1 - 5)  
**Kelompok:** G15 - MessiLearning (IF3170 Inteligensi Artifisial)

---
Notebook ini berisi pengujian 5 algoritma Supervised Learning *from scratch* menggunakan data sintetis (selama fase paralel) maupun data bersih hasil `DataPipeline`, lengkap dengan komparasi metrik (Akurasi, Presisi, Recall, F1-Score, Waktu Eksekusi) terhadap `scikit-learn`.

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.utils import (
    set_global_seed,
    RANDOM_SEED,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    calculate_custom_cost,
    k_fold_split,
)
from src.dummy_generator import generate_clean_synthetic_data, generate_dtl_synthetic_data
from src.knn import KNN
from src.dtl import DecisionTree, export_tree_visualization
from src.logistic_regression import LogisticRegression, generate_loss_contour_video
from src.svm import SVM, plot_svm_margin_2d
from src.naive_bayes import NaiveBayes, export_bayesian_network_graph

set_global_seed(RANDOM_SEED)
X_dummy, y_dummy = generate_clean_synthetic_data(n_samples=500, n_features=6)
X_dtl_dummy, y_dtl_dummy = generate_dtl_synthetic_data(n_samples=500)
print(f"Dummy Clean Shape: {X_dummy.shape}, Dummy DTL Shape: {X_dtl_dummy.shape}")

Dummy Clean Shape: (500, 6), Dummy DTL Shape: (500, 4)


## 1. K-Nearest Neighbors (KNN) — Anggota 1
Komparasi `src.knn.KNN` vs `sklearn.neighbors.KNeighborsClassifier`

In [2]:
from sklearn.neighbors import KNeighborsClassifier
# TODO (Anggota 1): Evaluasi KNN from scratch vs scikit-learn

## 2. Decision Tree Learning (DTL) — Anggota 2
Komparasi `src.dtl.DecisionTree` vs `sklearn.tree.DecisionTreeClassifier`, Visualisasi Top-$N$ Tree, dan Post-Pruning

In [3]:
from sklearn.tree import DecisionTreeClassifier
# TODO (Anggota 2): Evaluasi DecisionTree from scratch vs scikit-learn & ekspor visualisasi

## 3. Logistic Regression — Anggota 3
Komparasi `src.logistic_regression.LogisticRegression` ($L_1, L_2$, Elastic Net + Adam) vs `sklearn.linear_model.LogisticRegression` & Video Lintasan Loss

In [4]:
from sklearn.linear_model import LogisticRegression as SklearnLogReg
# TODO (Anggota 3): Evaluasi LogisticRegression from scratch vs scikit-learn & render video kontur

## 4. Support Vector Machine (SVM) — Anggota 4
Komparasi `src.svm.SVM` (Linear, Polynomial, Gaussian RBF) vs `sklearn.svm.SVC` & Plot Bidang Pemisah 2D

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# 1. Split data sintetis (atau ganti dengan load data bersih dari data/processed/ bila perlu)
X_tr_svm, X_va_svm, y_tr_svm, y_va_svm = train_test_split(
    X_dummy,
    y_dummy,
    test_size=0.2,
    stratify=y_dummy,
    random_state=RANDOM_SEED,
)

kernels = ["linear", "poly", "rbf"]
results = []

for kernel_name in kernels:
    # --- From-scratch SVM (cvxopt) ---
    t0 = time.perf_counter()
    scratch_svm = SVM(
        C=1.0,
        kernel=kernel_name,
        gamma="scale" if kernel_name != "linear" else None,
        random_state=RANDOM_SEED,
    )
    # Catatan: di SVM from-scratch, jika gamma=None maka dipakai 1/n_features;
    # set gamma=1.0 / X_tr_svm.shape[1] agar setara dengan 'scale'/standar
    scratch_svm = SVM(
        C=1.0,
        kernel=kernel_name,
        gamma=1.0 / X_tr_svm.shape[1],
        random_state=RANDOM_SEED,
    )
    scratch_svm.fit(X_tr_svm, y_tr_svm)
    y_pred_scratch = scratch_svm.predict(X_va_svm)
    scratch_time = time.perf_counter() - t0

    results.append({
        "Kernel": kernel_name,
        "Model": "Scratch SVM",
        "Accuracy": accuracy_score(y_va_svm, y_pred_scratch),
        "Precision": precision_score(y_va_svm, y_pred_scratch),
        "Recall": recall_score(y_va_svm, y_pred_scratch),
        "F1": f1_score(y_va_svm, y_pred_scratch),
        "Time (s)": round(scratch_time, 4),
    })

    # --- scikit-learn SVC ---
    t0 = time.perf_counter()
    sk_svm = SVC(
        C=1.0,
        kernel=kernel_name,
        gamma=1.0 / X_tr_svm.shape[1],
        random_state=RANDOM_SEED,
    )
    sk_svm.fit(X_tr_svm, y_tr_svm)
    y_pred_sk = sk_svm.predict(X_va_svm)
    sk_time = time.perf_counter() - t0

    results.append({
        "Kernel": kernel_name,
        "Model": "sklearn SVC",
        "Accuracy": accuracy_score(y_va_svm, y_pred_sk),
        "Precision": precision_score(y_va_svm, y_pred_sk),
        "Recall": recall_score(y_va_svm, y_pred_sk),
        "F1": f1_score(y_va_svm, y_pred_sk),
        "Time (s)": round(sk_time, 4),
    })

df_svm_results = pd.DataFrame(results)
display(df_svm_results)

# 2. Plot bidang pemisah 2D untuk ketiga kernel (menggunakan 2 fitur pertama)
X_2d_tr = X_tr_svm[:, :2]
plot_targets = [
    ("linear", "../output/plots/svm_margin_linear.png"),
    ("poly", "../output/plots/svm_margin_poly.png"),
    ("rbf", "../output/plots/svm_margin_rbf.png"),
]

for kernel_name, out_file in plot_targets:
    svm_2d = SVM(
        C=1.0,
        kernel=kernel_name,
        gamma=1.0 / X_2d_tr.shape[1],
        random_state=RANDOM_SEED,
    )
    svm_2d.fit(X_2d_tr, y_tr_svm)
    saved_path = plot_svm_margin_2d(
        svm_2d,
        X_2d_tr,
        y_tr_svm,
        output_path=out_file,
        title=f"SVM Decision Boundary & Margin ({kernel_name.upper()})",
    )
    print(f"Tersimpan: {saved_path}")

,Kernel,Model,Accuracy,Precision,Recall,F1,Time (s)
0,linear,Scratch SVM,0.93,0.905660,0.96,0.932039,0.0533
1,linear,sklearn SVC,0.93,0.905660,0.96,0.932039,0.0037
2,poly,Scratch SVM,0.91,0.901961,0.92,0.910891,0.0385
3,poly,sklearn SVC,0.93,0.890909,0.98,0.933333,0.0025
4,rbf,Scratch SVM,0.92,0.903846,0.94,0.921569,0.0364
5,rbf,sklearn SVC,0.92,0.903846,0.94,0.921569,0.0032


Tersimpan: ../output/plots/svm_margin_linear.png
Tersimpan: ../output/plots/svm_margin_poly.png
Tersimpan: ../output/plots/svm_margin_rbf.png


## 5. Naive Bayes & TAN — Anggota 5
Komparasi `src.naive_bayes.NaiveBayes` vs `sklearn.naive_bayes.GaussianNB` & Visualisasi Graf DAG Bayesian Network

In [6]:
from sklearn.naive_bayes import GaussianNB
# TODO (Anggota 5): Evaluasi NaiveBayes / TAN from scratch vs scikit-learn & simpan graf BN